## Use GimmeMotifs to find de novo motifs from scATAC-seq datasets

- last updated: 1/21/2025


In [1]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import the custom module
import sys
sys.path.append("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/zebrahub-multiome-analysis/scripts/gimmemotifs")

from atac_seq_motif_analysis import ATACSeqMotifAnalysis

INFO:matplotlib.font_manager:Failed to extract font properties from /usr/share/fonts/google-noto-emoji/NotoColorEmoji.ttf: In FT2Font: Can not load face (unknown file format; error code 0x2)


In [2]:
# Initialize the analysis class
genomes_dir = "/hpc/reference/sequencing_alignment/alignment_references/zebrafish_genome_GRCz11/fasta/"  # Replace with your genome data directory
ref_genome = "" # "danRer11" for zebrafish, GRCz11

atac_analysis = ATACSeqMotifAnalysis(ref_genome, genomes_dir)

In [3]:
# import the peak data
adata_peaks = sc.read_h5ad("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/01_Signac_processed/integrated_RNA_ATAC_counts_peaks_integrated.h5ad")
adata_peaks

AnnData object with n_obs × n_vars = 95196 × 640834
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'global_annotation', 'prediction.score.Lateral_Mesoderm', 'prediction.score.Neural_Crest', 'prediction.score.Somites', 'prediction.score.Epidermal', 'prediction.score.Neural_Anterior', 'prediction.score.Neural_Posterior', 'prediction.score.Endoderm', 'prediction.score.PSM', 'prediction.score.Differentiating_Neurons', 'prediction.score.Adaxial_Cells', 'prediction.score.NMPs', 'prediction.score.Notochord', 'prediction.score.Muscle', 'prediction.score.unassigned', 'prediction.score.max', 'nCount_peaks_bulk', 'nFeature_peaks_bulk', 'nCount_peaks_celltype', 'nFeature_peaks_celltype', 'nCount_peaks_merged', 'nFeature_peaks_merged', 'SCT.weight', 'peaks_merged.weight', 'nCount_Gene.Activity', 'nFeature_Gene.Activity', 'nCount_peaks_integrated', 'nF

In [5]:
adata_peaks.var_names

Index(['1-32-526', '1-2372-3057', '1-3427-4032', '1-4469-7268', '1-9541-9969',
       '1-11007-12962', '1-13276-13705', '1-14059-14260', '1-14625-15105',
       '1-15724-15934',
       ...
       '25-37489449-37490741', '25-37492035-37493157', '25-37493175-37493740',
       '25-37496420-37496948', '25-37497049-37497789', '25-37498106-37500090',
       '25-37500598-37500859', '25-37501104-37501839', 'MT-22-3567',
       'MT-13233-16532'],
      dtype='object', length=640834)

In [6]:
# Example list of peaks
peaks = ["1-32-526", "25-37492035-37493157"]

# Step 1: Convert peak strings to DataFrame
peaks_df = atac_analysis.list_peakstr_to_df(peaks)
print("Peaks DataFrame:")
print(peaks_df)



Peaks DataFrame:
  chr     start       end
0   1        32       526
1  25  37492035  37493157


In [23]:
# let's take all the peaks
peaks = adata_peaks.var_names.to_list()

# Step 1: Convert peak strings to DataFrame
peaks_df = atac_analysis.list_peakstr_to_df(peaks)
print("Peaks DataFrame:")
print(peaks_df.head())


Peaks DataFrame:
  chr  start   end
0   1     32   526
1   1   2372  3057
2   1   3427  4032
3   1   4469  7268
4   1   9541  9969


In [18]:
def check_peak_format(self, peaks_df):
    """
    Validate peak format and filter invalid peaks based on genome information.

    Args:
        peaks_df (pd.DataFrame): DataFrame with columns ["chr", "start", "end"].

    Returns:
        pd.DataFrame: Filtered DataFrame with valid peaks.
    """
    n_peaks_before = peaks_df.shape[0]
    all_chr_list = list(self.genome_data.keys())

    # Check chromosome names and peak lengths
    lengths = np.abs(peaks_df["end"] - peaks_df["start"])
    n_threshold = 5
    valid_peaks = peaks_df[(lengths >= n_threshold) & peaks_df["chr"].isin(all_chr_list)]

    # Check if peaks exceed chromosome lengths
    for idx, row in peaks_df.iterrows():
        chr_length = len(self.genome_data[row["chr"]])  # Fixed here
        if row["end"] > chr_length:
            valid_peaks = valid_peaks.drop(idx)

    # Print summary
    n_invalid_length = len(lengths[lengths < n_threshold])
    n_invalid_chr = n_peaks_before - peaks_df["chr"].isin(all_chr_list).sum()
    n_invalid_end = n_peaks_before - valid_peaks.shape[0]
    print(f"Peaks before filtering: {n_peaks_before}")
    print(f"Invalid chromosome names: {n_invalid_chr}")
    print(f"Invalid lengths (< {n_threshold} bp): {n_invalid_length}")
    print(f"Peaks exceeding chromosome lengths: {n_invalid_end}")
    print(f"Peaks after filtering: {valid_peaks.shape[0]}")

    return valid_peaks

In [24]:
# Step 2: Validate peaks
valid_peaks = check_peak_format(atac_analysis, peaks_df)
print("\nValid Peaks DataFrame:")
print(valid_peaks)

Peaks before filtering: 640834
Invalid chromosome names: 0
Invalid lengths (< 5 bp): 0
Peaks exceeding chromosome lengths: 3
Peaks after filtering: 640831

Valid Peaks DataFrame:
       chr     start       end
0        1        32       526
1        1      2372      3057
2        1      3427      4032
3        1      4469      7268
4        1      9541      9969
...     ..       ...       ...
640829  25  37498106  37500090
640830  25  37500598  37500859
640831  25  37501104  37501839
640832  MT        22      3567
640833  MT     13233     16532

[640831 rows x 3 columns]


In [27]:
# Step 3: Convert valid peaks to FASTA
fasta = atac_analysis.peak_to_fasta(valid_peaks["chr"] + "-" + valid_peaks["start"].astype(str) + "-" + valid_peaks["end"].astype(str))
fasta

640831 sequences

In [29]:
# Step 4: Remove zero-length sequences
filtered_fasta = atac_analysis.remove_zero_seq(fasta)


In [31]:
# Step 5: Save FASTA file
output_fasta_path = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/12_peak_umap_motif_analysis/peaks_integrated.fasta"

# save
with open(output_fasta_path, "w") as f:
    for name, seq in zip(filtered_fasta.ids, filtered_fasta.seqs):
        f.write(f">{name}\n{seq}\n")

## Use GimmeMotifs to find de novo motifs

In [ ]:
# from gimmemotifs.scanner import Scanner
# from gimmemotifs.fasta import Fasta
# from gimmemotifs.confg import DIRECT_NAME, INDIRECT_NAME


In [33]:
from gimmemotifs.denovo import gimme_motifs
help(gimme_motifs)

Help on function gimme_motifs in module gimmemotifs.denovo:

gimme_motifs(inputfile, outdir, params=None, filter_significant=True, cluster=True, create_report=True)
    De novo motif prediction based on an ensemble of different tools.
    
    Parameters
    ----------
    inputfile : str
        Filename of input. Can be either BED, narrowPeak or FASTA.
    
    outdir : str
        Name of output directory.
    
    params : dict, optional
        Optional parameters.
    
    filter_significant : bool, optional
        Filter motifs for significance using the validation set.
    
    cluster : bool, optional
        Cluster similar predicted (and significant) motifs.
    
    create_report : bool, optional
        Create output reports (both .txt and .html).
    
    Returns
    -------
    motifs : list
        List of predicted motifs.
    
    Examples
    --------
    
    >>> from gimmemotifs.denovo import gimme_motifs
    >>> gimme_motifs("input.fa", "motifs.out")



In [38]:
import celloracle as co
from celloracle import motif_analysis as ma
ref_genome = "danRer11"

genome_installation = ma.is_genome_installed(ref_genome=ref_genome,
                                             genomes_dir=None)
genome_installation

True

In [39]:
ma.SUPPORTED_REF_GENOME

,species,ref_genome,provider
0,Human,hg38,UCSC
1,Human,hg19,UCSC
2,Mouse,mm39,UCSC
3,Mouse,mm10,UCSC
4,Mouse,mm9,UCSC
5,S.cerevisiae,sacCer2,UCSC
6,S.cerevisiae,sacCer3,UCSC
7,Zebrafish,danRer7,UCSC
8,Zebrafish,danRer10,UCSC
9,Zebrafish,danRer11,UCSC


In [46]:
from gimmemotifs.config import MotifConfig

config = MotifConfig()
config.set_motif_dir("/hpc/mydata/yang-joon.kim/genomes/danRer11/")
config.set_bg_dir("/hpc/mydata/yang-joon.kim/genomes/danRer11/")
config.write(open(config.user_config, "w"))

print(f"Motif database directory: {config.get_motif_dir()}")
print(f"Background directory: {config.get_bg_dir()}")

Motif database directory: /hpc/mydata/yang-joon.kim/genomes/danRer11/
Background directory: /hpc/mydata/yang-joon.kim/genomes/danRer11/


In [68]:
import os
# Add the directory containing bedtools to PATH
os.environ["PATH"] += ":/home/yang-joon.kim/.conda/envs/celloracle_env/bin"
# Verify if BEDTools is now accessible
bedtools_path = os.popen("which bedtools").read().strip()
if bedtools_path:
    print(f"BEDTools found at: {bedtools_path}")
else:
    print("BEDTools is still not in PATH. Double-check the installation.")

BEDTools found at: /home/yang-joon.kim/.conda/envs/celloracle_env/bin/bedtools


In [71]:
import subprocess

try:
    subprocess.run(['bedtools', '--version'], check=True, capture_output=True)
    print("BEDTools is properly installed")
except subprocess.CalledProcessError as e:
    print(f"Error running bedtools: {e}")
except FileNotFoundError:
    print("BEDTools is not found in PATH")

BEDTools is properly installed


In [72]:
# from gimmemotifs.denovo import gimme_motifs

# Paths to input and output
# peaks_fasta = "/hpc/mydata/yang-joon.kim/genomes/danRer11/danRer11.fa"
# output_dir = "/path/to/output_dir"
peaks_fasta = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/12_peak_umap_motif_analysis/peaks_integrated.fasta"
output_dir = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/12_peak_umap_motif_analysis/"

# Parameters
params = {
    "genome": "danRer11",  # Specify the genome name
    "tools": "Homer,BioProspector,MEME",
    "size": 200,
}

# Run GimmeMotifs
motifs = gimme_motifs(peaks_fasta, output_dir, params=params)

DEBUG:gimme.config:Using multiprocessing
DEBUG:gimme.config:Parameters:
DEBUG:gimme.config:  fraction: 0.2
DEBUG:gimme.config:  use_strand: False
DEBUG:gimme.config:  abs_max: 1000
DEBUG:gimme.config:  analysis: xl
DEBUG:gimme.config:  enrichment: 1.5
DEBUG:gimme.config:  size: 200
DEBUG:gimme.config:  lsize: 500
DEBUG:gimme.config:  background: ['gc', 'random']
DEBUG:gimme.config:  cluster_threshold: 0.95
DEBUG:gimme.config:  scan_cutoff: 0.9
DEBUG:gimme.config:  available_tools: AMD,BioProspector,ChIPMunk,HMS,Homer,Improbizer,MDmodule,MotifSampler,Posmo
DEBUG:gimme.config:  tools: Homer,BioProspector,MEME
DEBUG:gimme.config:  pvalue: 0.001
DEBUG:gimme.config:  max_time: -1
DEBUG:gimme.config:  ncpus: 12
DEBUG:gimme.config:  motif_db: gimme.vertebrate.v5.0.pfm
DEBUG:gimme.config:  use_cache: False
DEBUG:gimme.config:  genome: danRer11
DEBUG:gimme.config:No time limit for motif prediction
2025-01-21 16:24:59,641 - INFO - starting full motif analysis
INFO:gimme.denovo:starting full moti

NotImplementedError: "windowMaker" does not appear to be installed or on the path, so this method is disabled.  Please install a more recent version of BEDTools and re-import to use this method.

## Testing the gimmemotifs package with the tutorial/sample dataset


In [76]:
# Or from a consensus sequence
from gimmemotifs.motif import motif_from_consensus
ap1 = motif_from_consensus("TGASTCA")
print(ap1.to_ppm())

>TGASTCA
0.0000	0.0000	0.0000	1.0000
0.0000	0.0000	1.0000	0.0000
1.0000	0.0000	0.0000	0.0000
0.0000	0.5000	0.5000	0.0000
0.0000	0.0000	0.0000	1.0000
0.0000	1.0000	0.0000	0.0000
1.0000	0.0000	0.0000	0.0000


In [79]:
# MEME
print(ap1.to_meme())

MOTIF TGASTCA
BL   MOTIF TGASTCA width=0 seqs=0
letter-probability matrix: alength= 4 w= 7 nsites= 1200.0 E= 0
0.0	0.0	0.0	1.0
0.0	0.0	1.0	0.0
1.0	0.0	0.0	0.0
0.0	0.5	0.5	0.0
0.0	0.0	0.0	1.0
0.0	1.0	0.0	0.0
1.0	0.0	0.0	0.0
